In [ ]:
import numpy as np


In [ ]:
WT = np.load("final_embeddings/1A5E_00_WT.npy", allow_pickle=True)
#mut = np.load("1A5_01_A73E.npy", allow_pickle=True)

print(WT)

In [ ]:
import pathlib as pt

path = pt.Path("final_embeddings")

file_list =  []

for file in path.iterdir():
    file = str(file)
    file = file.split(".")[0]
    file = file.split("\\")[1]
    file = file.split("_")
    file = file[0] + "_" + file[2]
    file_list.append(file)
    



In [ ]:
from pathlib import Path

path = Path("final_embeddings")

file_list = [
    f"{p.stem.split('_')[0]}_{p.stem.split('_')[2]}"
    for p in path.iterdir()
    if p.is_file()
]

len(file_list)

In [ ]:
import os
from pathlib import Path

path = Path("final_embeddings")

file_list_ext = [x+".npy" for x in file_list]

for file, new_name in zip(path.iterdir(), file_list_ext):
    dst = path / new_name   # full destination path

    # Remove existing file if it has the same name
    if dst.exists():
        dst.unlink()        # same as os.remove but works with Path objects

    file.rename(dst)

# preparing the pikle file

In [ ]:
import pandas as pd

df = pd.read_csv("conversion_tracker.csv", header = 0, sep = ",")

print(len(df))
df.iloc[30]



In [ ]:
df["file_name"] = df.original_protein.astype(str) + "_" + df.mutation.astype(str) + ".npy"
df = df[["file_name", "fitness"]]

print(len(df))
df.iloc[30]


In [ ]:
folder_path = Path("final_embeddings")

# Get unique file names without extension
unique_files = list(f.stem +".npy" for f in folder_path.iterdir())

rows_list = []

for file in unique_files:

    matching_rows = df[df["file_name"] == file]  # exact match

    if not matching_rows.empty:

        if len(matching_rows) > 1:
            matching_rows = matching_rows.iloc[[0]]

        rows_list.append(matching_rows)

# Combine all matching rows into a new DataFrame
df = pd.concat(rows_list, ignore_index=True)


print(len(df))
df.head(3)


In [ ]:
# Assuming df["file_name"] contains the file names
missing_in_df = set(unique_files) - set(df["file_name"])

print("Files in folder but not in DataFrame:")
print(missing_in_df)



In [ ]:

# Remove file extension and split into prefix and suffix
df[['prefix', 'suffix']] = df['file_name'].str.replace('.npy', '', regex=False).str.split('_', expand=True)

# Count WT occurrences per prefix
wt_counts = df[df['suffix'] == 'WT'].groupby('prefix').size()

# If wt_counts is a Series (as in previous example)
print("Number of prefixes:", len(wt_counts))
print("Total WT files:", wt_counts.sum())
print("Total number of mutation files:", len(df) - wt_counts.sum())


In [ ]:
import numpy as np
from pathlib import Path

print_df = df[["file_name", "fitness"]].copy()

emb_path = Path("final_embeddings")

# Preload all needed .npy files into a dictionary
embedding_dict = {
    fname: np.load(emb_path / (fname))
    for fname in print_df["file_name"]
}

# Add the arrays to the dataframe
print_df["embedding"] = print_df["file_name"].map(embedding_dict)
print_df.to_pickle("embeddings.pkl")
